# Chapter 5 — Agent Evaluation & Cost Controls (v2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IvanReznikov/LangChain4LifeSciencesHealthcare)

**Learning objectives**
- Measure tool-call validity and groundedness
- Track token usage and estimate cost per run
- Enforce reproducibility with seeds and logging
- Build a small agent eval harness

> Runtime: ~12 min (API)  
> Cost: paid LLM required  
> Data: synthetic eval cases


> **LangChain 1.x (2026)** — built on `langchain-core==1.2.30`, `langchain==1.0.0`. See `UPDATE_2026.md`.


## Environment setup


### Secrets (Colab or local)


In [7]:
# @title Setting environmental variables
import os

# --- Dual-mode secrets: works in Google Colab AND locally (.env / environment) ---
try:
    from google.colab import userdata  # type: ignore

    IN_COLAB = True
except Exception:
    userdata = None
    IN_COLAB = False

if not IN_COLAB:
    # Local run: load variables from a .env file if present (never commit .env!).
    try:
        from dotenv import load_dotenv

        load_dotenv()
    except Exception:
        pass


def get_secret(name, default=None):
    """Read a secret from Colab Secrets, else from local env/.env, else default."""
    if IN_COLAB and userdata is not None:
        try:
            val = userdata.get(name)
            if val:
                return val
        except Exception:
            pass
    return os.getenv(name, default)


# 👇 Choose your provider 👇
API_KEY_PROVIDER = "OPENAI"  # "GEMINI" | "OPENAI" | "GROQ" | "ANTHROPIC"

if API_KEY_PROVIDER == "OPENAI":
    os.environ["OPENAI_API_KEY"] = get_secret("LC4LSH_OPENAI_API_KEY", "sk-...")
elif API_KEY_PROVIDER == "ANTHROPIC":
    os.environ["ANTHROPIC_API_KEY"] = get_secret(
        "LC4LSH_ANTHROPIC_API_KEY", "sk-ant-..."
    )
elif API_KEY_PROVIDER == "GEMINI":
    os.environ["GOOGLE_API_KEY"] = get_secret("LC4LSH_GOOGLE_API_KEY", "AIza...")
elif API_KEY_PROVIDER == "GROQ":
    os.environ["GROQ_API_KEY"] = get_secret("LC4LSH_GROQ_API_KEY", "gsk_...")

print(
    f"✅ API keys loaded for {API_KEY_PROVIDER} (source: {'Colab Secrets' if IN_COLAB else 'local env/.env'})"
)

# Hugging Face token (optional; needed for gated models)
os.environ["HF_TOKEN"] = get_secret("HF_TOKEN", "") or ""

API keys loaded for OPENAI


### Install pinned dependencies


In [8]:
# @title Installing Python dependencies
%pip install -q "langchain==1.0.0" "langchain-core==1.2.30" "langchain-openai==1.0.0" "langchain-community==0.4.0" "langgraph>=0.2" "pydantic>=2.5" python-dotenv
# Pinned versions - Last validated: 2026-07-21 (see UPDATE_2026.md)

In [9]:
# @title Setting LangSmith variables
# ========================
# 👇 CONFIGURE HERE 👇
# ========================
LANGSMITH_API_KEY = userdata.get("LANGSMITH_API_KEY") or "lsv2_pt_..."
LANGSMITH_PROJECT = "lc4lsh-chapter5-agent-eval"  # Traces appear under this name
REGION = "EU"  # "EU" or "US" - must match your account!
# ========================

if LANGSMITH_API_KEY and LANGSMITH_API_KEY != "lsv2_pt_...":
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
    os.environ["LANGSMITH_ENDPOINT"] = (
        "https://eu.api.smith.langchain.com"
        if REGION == "EU"
        else "https://api.smith.langchain.com"
    )
    dashboard = (
        "https://eu.smith.langchain.com"
        if REGION == "EU"
        else "https://smith.langchain.com"
    )
    print(f"✅ LangSmith enabled!")
    print(f"   Region: {REGION} | Project: {LANGSMITH_PROJECT}")
    print(f"   Dashboard: {dashboard}")
else:
    print("⚠️ LangSmith disabled - paste your API key above to enable tracing")
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith OFF")

✅ LangSmith enabled!
   Region: EU | Project: lc4lsh-chapter5-agent-eval
   Dashboard: https://eu.smith.langchain.com


## Why evaluate agents?

An agent can look right while calling the wrong tools, hallucinating evidence, or burning budget. We evaluate four things:

- **Tool-call validity** — did it call real tools with well-formed args?
- **Groundedness** — is the answer supported by retrieved evidence?
- **Reproducibility** — same input + seed → same output?
- **Cost** — tokens and dollars per run


## 1. A run record for every agent execution


In [10]:
from pydantic import BaseModel, Field
import time


class RunRecord(BaseModel):
    question: str
    tool_calls: list[dict] = []
    answer: str = ""
    prompt_tokens: int = 0
    completion_tokens: int = 0
    latency_s: float = 0.0

    @property
    def total_tokens(self):
        return self.prompt_tokens + self.completion_tokens

    def cost_usd(self, in_per_1k=0.00015, out_per_1k=0.0006):
        return (self.prompt_tokens / 1000) * in_per_1k + (
            self.completion_tokens / 1000
        ) * out_per_1k


print("RunRecord ready")

RunRecord ready


## 2. A tiny agent that records its run


In [11]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool


@tool("get_temp")
def get_temp(city: str) -> str:
    """Get a (fake) temperature for a city."""
    return f"{city}: 22C"


llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_tools = llm.bind_tools([get_temp])


def run_agent(question):
    rec = RunRecord(question=question)
    t0 = time.time()
    msg = llm_tools.invoke(question)
    rec.latency_s = time.time() - t0
    usage = getattr(msg, "usage_metadata", None) or {}
    rec.prompt_tokens = usage.get("input_tokens", 0)
    rec.completion_tokens = usage.get("output_tokens", 0)
    for tc in getattr(msg, "tool_calls", []) or []:
        rec.tool_calls.append({"name": tc["name"], "args": tc["args"]})
        if tc["name"] == "get_temp":
            rec.answer = get_temp.invoke(tc["args"])
    if not rec.answer:
        rec.answer = msg.content
    return rec


rec = run_agent("What is the temperature in Basel?")
print(rec.model_dump())

{'question': 'What is the temperature in Basel?', 'tool_calls': [{'name': 'get_temp', 'args': {'city': 'Basel'}}], 'answer': 'Basel: 22C', 'prompt_tokens': 54, 'completion_tokens': 15, 'latency_s': 1.0752670764923096}


## 3. Metric: tool-call validity


In [12]:
KNOWN_TOOLS = {"get_temp"}


def tool_call_validity(rec):
    if not rec.tool_calls:
        return None
    valid = sum(
        1
        for tc in rec.tool_calls
        if tc["name"] in KNOWN_TOOLS and isinstance(tc["args"], dict)
    )
    return valid / len(rec.tool_calls)


print("Tool-call validity:", tool_call_validity(rec))

Tool-call validity: 1.0


## 4. Metric: groundedness (evidence support)


In [13]:
def groundedness(rec, evidence):
    """Fraction of answer key tokens that appear in evidence (toy proxy)."""
    ans_words = set(rec.answer.lower().split())
    ev_words = set(evidence.lower().split())
    if not ans_words:
        return 0.0
    return len(ans_words & ev_words) / len(ans_words)


print("Groundedness:", round(groundedness(rec, "Basel: 22C"), 2))

Groundedness: 1.0


## 5. Reproducibility check


In [14]:
def reproducible(question, n=2):
    answers = {run_agent(question).answer for _ in range(n)}
    return len(answers) == 1, answers


ok, answers = reproducible("What is the temperature in Basel?")
print("Reproducible:", ok)

Reproducible: True


## 6. Cost report


In [15]:
print(
    f"prompt={rec.prompt_tokens} completion={rec.completion_tokens} total={rec.total_tokens}"
)
print(f"latency={rec.latency_s:.2f}s  est_cost=${rec.cost_usd():.6f}")

prompt=54 completion=15 total=69
latency=1.08s  est_cost=$0.000017


## Limitations & safety notes

- Groundedness here is a token-overlap proxy, not a real faithfulness judge; use an LLM-judge or RAGAS for production.
- Cost constants are illustrative; check current pricing.
- Reproducibility needs `temperature=0` and may still vary across model versions.
- **Paid API required**.


---
### Further Reading

| Notebook | Relevance |
|----------|-----------|
| **Chapter 5 Building Personal Assistants LangGraph and Agents** | The agent being evaluated |
| **Chapter 10 Evaluation CI** | CI-ready evaluation pipeline |


In [16]:
# Cleanup
import gc

for _v in ("llm", "model", "agent", "graph", "app", "workflow"):
    globals().pop(_v, None)
gc.collect()
print("Cleanup complete.")

Cleanup complete.


## Exercises

<details><summary>Why track tool-call validity?</summary>Malformed or hallucinated tool calls are a leading agent failure mode; measuring them catches it early.</details>

<details><summary>Why is token-overlap only a proxy for groundedness?</summary>An answer can share words yet be unsupported, or be supported with different wording; a judge model is more accurate.</details>

<details><summary>What two things drive reproducibility?</summary>Deterministic decoding (temperature=0/seed) and pinned model+library versions.</details>

### Tasks
- **Task A** - Add a `max_cost_usd` guard that raises before a run would exceed budget.
- **Task B** - Replace the proxy groundedness with a gated LLM judge that returns supported/unsupported.
- **Task C** - Log runs to JSONL and compute average cost + validity over 5 questions.
- **Task D** - Add a regression: assert `tool_call_validity == 1.0` in a test, and show a failing case.
